In [0]:
%pip install seaborn

In [0]:
import os
import sys
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
from pyspark.sql import functions as F

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(repo_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from data_health_ftw.common import config
from data_health_ftw.ml import dataset
from data_health_ftw.ml.dataset import build_modeling_frame

In [0]:
df_ml = (
    spark.table("workspace.gold.ml_features")
    .filter("year >= 2010")
    .toPandas()
)

display(df_ml.round(2))

In [0]:
if hasattr(df_ml, "toPandas"):
    ml_pdf = df_ml.toPandas()
else:
    ml_pdf = df_ml

In [0]:
modeling_frame = build_modeling_frame(
    ml_pdf,
    features=config.ML_FEATURES,
    targets=config.ML_TARGETS,
    min_year=config.ML_MIN_YEAR,
    fill_from_past_columns=config.ML_FILL_FROM_PAST,
)

In [0]:
rows_country = (
    spark.table("workspace.gold.ml_features")
    .groupBy("country_code")
    .count()
)

In [0]:
%sql
SELECT year, COUNT(*) AS row_count, 
COUNT(DISTINCT country_code) AS country_count
FROM workspace.gold.ml_features
GROUP BY year
ORDER BY year;

In [0]:
feature_cols = list(config.ML_FEATURES)

In [0]:
df_ml.shape

In [0]:
print(df_ml.columns.tolist())

In [0]:
RANDOM_STATE = 42
N_SPLITS = 5

In [0]:
targets = ["cvd_asdr", "chronic_resp_asdr"]

In [0]:
print((df_ml[targets] <= 0).sum())
print(df_ml[targets].isna().sum())
print(df_ml[targets].describe().T[["mean", "50%", "min", "max"]])

In [0]:
repo_root

In [0]:
cols_with_nulls = df_ml.columns[df_ml.isna().any()]

In [0]:
null_rate = df_ml[cols_with_nulls].isna().groupby(df_ml["year"]).mean().T

In [0]:
plt.figure(figsize=(10, 9))
ax = sns.heatmap(
    null_rate,
    annot=True,
    fmt=".1%",
    cmap="RdYlGn_r",
    cbar_kws={"format": PercentFormatter(xmax=1, decimals=0)},
)
    
plt.show()

In [0]:
features = list(config.ML_FEATURES)
targets = list(config.ML_TARGETS)

In [0]:
fig, axes = plt.subplots(2, len(targets), figsize=(5 * len(targets), 7))
for j, target in enumerate(targets):
    axes[0, j].hist(df_ml[target], bins=40)
    axes[0, j].set_title(f"{target} (raw)")
    axes[1, j].hist(np.log1p(df_ml[target]), bins=40)
    axes[1, j].set_title(f"log1p({target})")
plt.tight_layout()
plt.show()
 
df_ml[targets].describe().round(1)

In [0]:
for target in targets:
    y = np.log1p(df_ml[target])
    total = ((y - y.mean()) ** 2).sum()
    by_country = ((df_ml.assign(y=y).groupby("country_code")["y"].transform("mean") - y.mean()) ** 2).sum()
    by_year = ((df_ml.assign(y=y).groupby("year")["y"].transform("mean") - y.mean()) ** 2).sum()
    print(f"{target:20s} country={by_country / total:5.1%}  year={by_year / total:5.1%}")

In [0]:
df_ml.select_dtypes(include=["number"]).corr()

In [0]:
country_with_null = sum(F.col())